<a href="https://colab.research.google.com/github/sivasooryagiri/Learn-PyTorch-for-deep-learning-in-a-day--Notes/blob/main/05_GoingModular_self.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
source = "https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip"

In [ ]:
%%writefile going_modular/get_data.py

"""Check and download data pizza_steak_sushi for training"""
import os
import requests
import zipfile
from pathlib import Path
def download_data():
  # Setup path to data folder
  data_path = Path("data/")
  image_path = data_path / "pizza_steak_sushi"

  # If image folder doesen't exist, download and prepare it

  if data_path.is_dir():
    print(f"{image_path} directory already exists.")

  else:
    print(f"Did not find {image_path} directory, creating one.")
    image_path.mkdir(parents=True, exist_ok=True)

    with open(data_path / "pizza_steak_sushi.zip", "wb") as f:
      request = requests.get(source)
      print("Downlading the file ....")
      f.write(request.content)

    with zipfile.ZipFile(data_path / "pizza_steak_sushi.zip","r") as zip_ref:
      print("unziping the files...")
      zip_ref.extractall(image_path)

    os.remove(data_path/ "pizza_steak_sushi.zip")

Overwriting going_modular/get_data.py


In [ ]:
os.makedirs("going_modular", exist_ok=True)

In [ ]:
%%writefile going_modular/data_setup.py
"""
Contains functionality for creating Pytorch Dataloaders for
image classification data.
"""
import os

from torchvision import transforms, datasets
from torch.utils.data import DataLoader

NUM_WORKERS = os.cpu_count()

def create_dataloaders(
    train_dir: str,
    test_dir: str,
    transform: transforms.Compose,
    batch_size: int,
    num_workers: int=NUM_WORKERS
):
  """Create training and testing DataLoaders
  Args:
  train_dir: Path to training directory
  test_dir: Path to testing directory
  transforms: torchvision transforms to perform on training and testing data
  batch_size: Number of samples per batch in each of DataLoaders
  num_workers: An integer for number of workers per DataLoader

  Returns:
  A tuple of (train_dataloader, test_dataloader, class_names)
  """
  #Use Image folder to create datasets
  train_data = datasets.ImageFolder(train_dir, transform=transform)
  test_data = datasets.ImageFolder(test_dir, transform=transform)

  # getting Class Names
  class_names = train_data.classes

  # turn images into dataloaders
  train_dataloader = DataLoader(
      train_data,
      batch_size=batch_size,
      shuffle=True,
      num_workers=NUM_WORKERS,
      pin_memory=True
  )
  test_dataloader = DataLoader(
      test_data,
      batch_size=batch_size,
      shuffle=False,
      num_workers=NUM_WORKERS,
      pin_memory=True
  )

  return train_dataloader, test_dataloader, class_names

Writing going_modular/data_setup.py


In [ ]:
%%writefile going_modular/model_builder.py
"""
Contains PyTorch model code to instantiate an TinyVGG model.
"""

import torch
from torch import nn

class TinyVGG(nn.Module):
  """
  Creates the TinyVGG architecture.

  Replicates the TinyVGG architecture from the CNN explainer.
  Args:
  input_shape: An integer indicating number of input channels/
  hidden_units: An integer indicating number of hidden units between layers.
  output_shape: An integer indicating number of output units.
  """

  def __init__(self,
               input_shape: int,
               hidden_units: int,
               output_shape: int
               ):
    super().__init__()
    self.conv_block_1 = nn.Sequential(
        nn.Conv2d(
            in_channels=input_shape,
            out_channels=hidden_units,
            kernel_size=3,
            stride=1,
            padding=0),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=0),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,
                     stride=2)
    )
    self.conv_block_2 = nn.Sequential(
        nn.Conv2d(in_channels=hidden_units,
                  out_channels=hidden_units,
                  kernel_size=3,
                  padding=0),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units,
                  out_channels=hidden_units,
                  kernel_size=3,
                  padding=0),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,
                     stride=2)
    )
    self.classifier = nn.Sequential(
        nn.Flatten(),
        # 13*13 as all conv will reduce dim by 2 and maxpool will reduce aswell
        nn.Linear(in_features=hidden_units*13*13,
                  out_features=output_shape)
    )
  def forward(self, x:torch.Tensor):
    x = self.conv_block_1(x)
    x = self.conv_block_2(x)
    x = self.classifier(x)
    return x


Writing going_modular/model_builder.py


In [ ]:
%%writefile going_modular/engine.py
"""
Contain functions for training and testing a PyTorch model.
"""
import torch
from tqdm.auto import tqdm
from typing import Dict, List, Tuple

def train_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               device: torch.device):
  """Train a PyTorch model for a single epoch
  Turns a target PyTorch model to training mode and then
  runs through all of the required training steps (forward and
  pass, loss calculation, optimizer step)

  Args:
    model: A PyTorch model to be trained .
    dataloader: A Dataloader instance for the model to be trained on.
    loss_fn: A PyTorch loss function to minimize.
    optimizer: A PyTorch optimizer to help minimize the loss function.
    device: A target device to comute on

  Returns:
    A tuple of training loss and training accuracy metrics.
    In the form (train_loss, train_accuracy)"""

  # Put model to trainig mode
  model.train()
  train_loss, train_acc = 0, 0

  # Loop through data loader data batches
  for batch,(X,y) in enumerate(dataloader):
    # send data to target device
    X, y = X.to(device), y.to(device)

    # Forward pass
    y_pred = model(X)

    #Calculate and accumilate loss
    loss = loss_fn(y_pred,y)
    train_loss += loss.item()

    # Optimizer zero grad
    optimizer.zero_grad()

    # Loss Backward
    loss.backward()

    #optimizer step
    optimizer.step()

    y_pred_class = torch.argmax(torch.softmax(y_pred,dim=1), dim=1)
    train_acc += (y_pred_class == y).sum().item()/len(y_pred)
  # Adjust metrics to get average loss and accuracy per batch
  train_loss =train_loss / len(dataloader)
  train_acc = train_acc / len(dataloader)
  return train_loss, train_acc

def test_step(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module,
              device: torch.device):
  """Test a PyTorch model for a single epoch.
  Turns a target PyTorch modle to 'eval' mode and performs a forward pass on testing dataset"""
  model.eval()
  test_loss, test_acc = 0,0

  with torch.inference_mode():
    for batch, (X, y) in enumerate(dataloader):
      X, y = X.to(device), y.to(device)
      test_pred_logits = model(X)
      loss = loss_fn(test_pred_logits, y)
      test_loss += loss.item()
      test_pred_labels = test_pred_logits.argmax(dim=1)
      test_acc += ((test_pred_labels==y).sum().item()/len(test_pred_labels))

  test_loss = test_loss / len(dataloader)
  test_acc = test_acc / len(dataloader)

  return test_loss, test_acc

def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module,
          epochs: int,
          device: torch.device):
  """ Train and test a PyTorch model.
  Passes a targert PyTorch models through train_step()
  functions for a number of epochs, training and testing the model
  in the same epoch loop.

  Calculates, prints and stores evaluation metrics throughout.
  """
  results = {"train_loss":[],
             "train_acc":[],
             "test_loss":[],
             "test_acc":[]}
  for epoch in tqdm(range(epochs)):
    train_loss, train_acc = train_step(model=model,
                                       dataloader=train_dataloader,
                                       loss_fn=loss_fn,
                                       optimizer=optimizer,
                                       device=device)
    test_loss, test_acc = test_step(model=model,
                                    dataloader=test_dataloader,
                                    loss_fn=loss_fn,
                                    device=device)
    print(
        f"Epoch: {epoch+1} | "
        f"train_loss: {train_loss:.4f} | "
        f"test_loss: {test_loss:.4f} | "
        f"test_acc: {test_acc:.4f}"
    )

    # Update results dictionary
    results["train_loss"].append(train_loss)
    results["train_acc"].append(train_acc)
    results["test_loss"].append(test_loss)
    results["test_acc"].append(test_acc)

  return results


Writing going_modular/engine.py


In [ ]:
%%writefile going_modular/utils.py

import torch
from pathlib import Path

def save_model(model: torch.nn.Module,
               target_dir: str,
               model_name: str):

  """ Saves a PyTorch model to target directory.
  model_name: should end with ".pth" or ".pt" """
  # Create a target directory
  target_dir_path = Path(target_dir)
  target_dir_path.mkdir(parents=True, exist_ok=True)

  assert model_name.endswith(".pth") or model_name.endswith(".pt"),"model_name should end with 'pt' or '.pth' "
  model_save_path = target_dir_path / model_name

  # Save the model state_dict()
  print(f"[INFO] Saving a model to {model_save_path}")
  torch.save(obj=model.state_dict,
             f=model_save_path)

Writing going_modular/utils.py


In [ ]:
%%writefile going_modular/train.py
""" Trains a PyTorch image classification model using device-agnostic code."""
import os
import torch
import get_data, data_setup, engine, model_builder, utils

from torchvision import transforms

# Setup hyperparamerters
NUM_EPOCHS = 5
BATCH_SIZE = 32
HIDDEN_UNITS = 10
LEARNING_RATE = 0.001

# Downloading data from
get_data.download_data()
# Setup directories
train_dir = "data/pizza_steak_sushi/train"
test_dir = "data/pizza_steak_sushi/test"
# Setup target device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Create transforms
data_transform = transforms.Compose([
    transforms.Resize((64,64)),
    transforms.ToTensor()
])
# Creating DataLoaders
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=data_transform,
    batch_size=BATCH_SIZE
)
#Create models
model = model_builder.TinyVGG(
    input_shape=3,
    hidden_units=HIDDEN_UNITS,
    output_shape=len(class_names)
).to(device)

# Set loss and optimizer
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),
                             lr=LEARNING_RATE)
# Start training with help from engine.py
engine.train(model=model,
             train_dataloader=train_dataloader,
             test_dataloader=test_dataloader,
             loss_fn=loss_fn,
             optimizer = optimizer,
             epochs=NUM_EPOCHS,
             device=device
             )
utils.save_model(model=model,
                 target_dir="models",
                 model_name="05_going_moduler_script_mode_tinyvgg_model.pth")

Overwriting going_modular/train.py


In [ ]:
!python going_modular/train.py

data/pizza_steak_sushi directory already exists.
  0% 0/5 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Epoch: 1 | train_loss: 1.1035 | test_loss: 1.1183 | test_acc: 0.2604
 20% 1/5 [00:01<00:06,  1.65s/it]Epoch: 2 | train_loss: 1.1002 | test_loss: 1.1281 | test_acc: 0.1979
 40% 2/5 [00:03<00:04,  1.64s/it]Epoch: 3 | train_loss: 1.0886 | test_loss: 1.1341 | test_acc: 0.3333
 60% 3/5 [00:04<00:03,  1.64s/it]Epoch: 4 | train_loss: 1.0831 | test_loss: 1.1351 | test_acc: 0.2708
 80% 4/5 [00:06<00:01,  1.63s/it]Epoch: 5 | train_loss: 1.0421 | test_loss: 1.1237 | test_acc: 0.3229
100% 5/5 [00:09<00:00,  1.83s/it]
[INFO] Saving a model to models/05_going_moduler_script_mode_tinyvgg_model.pth


In [ ]:
import argparse
parser = argparse.ArgumentParser(description="Get some hyperparameters.")
parser.add_argument("--num_epochs",
                     default=10,
                     type=int,
                     help="the number of epochs to train for")

_StoreAction(option_strings=['--num_epochs'], dest='num_epochs', nargs=None, const=None, default=10, type=<class 'int'>, choices=None, required=False, help='the number of epochs to train for', metavar=None)